In [4]:
import pyterrier as pt
import requests
import pandas as pd
import pickle
import nltk
import numpy as np
import json

In [42]:
# load bibsonomy data, filter type = publication
url = "https://www.bibsonomy.org/json/search/database?items=1000"
result = requests.get(url)
data = result.json()
df = pd.DataFrame(data["items"])
df = df[df["type"] == "Publication"]

In [32]:
#drop columns where at least 50% of the rows have missing values
df_filtered = df.dropna(axis=1, thresh=len(df)*0.5)

In [34]:
df_filtered.columns

Index(['type', 'id', 'tags', 'intraHash', 'label', 'user', 'description',
       'date', 'changeDate', 'count', 'url', 'interHash', 'pub-type',
       'journal', 'year', 'author', 'authors', 'volume', 'bibtexKey', 'pages'],
      dtype='object')

In [38]:
useful_columns = ['id', 'tags', 'label', 'description', 'date', 'changeDate', 'url', 'pub-type', 'year', 'author', 'authors']
df_filtered = df_filtered[useful_columns]

In [40]:
df_filtered

,id,tags,label,description,date,changeDate,url,pub-type,year,author,authors
1000,https://www.bibsonomy.org/bibtex/2750b8770eec8...,"[ecoli, gene-networks, database]",EcoliNet: a database of cofunctional gene netw...,,2018-12-02 16:09:07,2018-12-02 16:09:07,http://dx.doi.org/10.1093/database/bav001,article,2015,"[Hanhae Kim, Jung Eun E. Shim, Junha Shin, Ins...","[{'first': 'Hanhae', 'last': 'Kim'}, {'first':..."
1001,https://www.bibsonomy.org/bibtex/2d93ac9b6cfa6...,"[database, drug-protein-interactions, protein-...",The drug-minded protein interaction database (...,,2018-12-02 16:09:07,2018-12-02 16:09:07,http://dx.doi.org/10.1093/database/baw041,article,2016,"[Meik Kunz, Chunguang Liang, Santosh Nilla, Al...","[{'first': 'Meik', 'last': 'Kunz'}, {'first': ..."
1002,https://www.bibsonomy.org/bibtex/233d3effaf95a...,"[database, phosphorylation, signalling, yeast]",PhosphoGRID: a database of experimentally veri...,,2018-12-02 16:09:07,2018-12-02 16:09:07,http://dx.doi.org/10.1093/database/bap026,article,2010,"[Chris Stark, Ting-Cheng C. Su, Ashton Breitkr...","[{'first': 'Chris', 'last': 'Stark'}, {'first'..."
1003,https://www.bibsonomy.org/bibtex/2ca6461cce248...,[test],GoSynthetic database tool to analyse natural a...,,2013-12-11 13:22:35,2013-12-11 13:22:35,http://dx.doi.org/10.1093/database/bat043,article,2013,"[C. Liang, B. Kruger, T. Dandekar]","[{'first': 'C.', 'last': 'Liang'}, {'first': '..."
1004,https://www.bibsonomy.org/bibtex/2eb1605409b13...,"[JSON, XML, Database, documents, NOSQL, SQL]",Database for Mobile Application,,2018-09-04 08:57:56,2018-10-02 11:01:26,http://www.ijtsrd.com/computer-science/databas...,article,2018,[Jasra Nisar Owais Noor Trumboo],"[{'first': 'Jasra Nisar Owais Noor', 'last': '..."
...,...,...,...,...,...,...,...,...,...,...,...
1995,https://www.bibsonomy.org/bibtex/24656463e6ea6...,[dblp],A Software Tool for Modular Database Design.,,2024-02-06 00:00:00,2024-04-09 06:31:39,http://dblp.uni-trier.de/db/journals/tods/tods...,article,1991,"[Marco A. Casanova, António L. Furtado, Luiz T...","[{'first': 'Marco A.', 'last': 'Casanova'}, {'..."
1996,https://www.bibsonomy.org/bibtex/28615de192f73...,[dblp],An Operating System for a Database Machine.,,2020-03-10 00:00:00,2020-03-11 11:39:24,http://dblp.uni-trier.de/db/journals/debu/debu...,article,1986,[Chris Nyberg],"[{'first': 'Chris', 'last': 'Nyberg'}]"
1997,https://www.bibsonomy.org/bibtex/262cf539f3759...,"[indikatoren, internationalität, entwicklung, ...",The utility of bibliographic databases as indi...,,2010-04-20 14:51:32,2010-04-20 14:51:32,http://www.sciencedirect.com/science/article/B...,article,1991,[Gretchen Whitney],"[{'first': 'Gretchen', 'last': 'Whitney'}]"
1998,https://www.bibsonomy.org/bibtex/2cb591ff67a93...,[dblp],Influenza sequence and epitope database.,,2020-05-17 00:00:00,2020-05-19 12:02:02,http://dblp.uni-trier.de/db/journals/nar/nar37...,article,2009,"[In Seok Yang, Joo-Yeon Lee, Joon Seung Lee, W...","[{'first': 'In Seok', 'last': 'Yang'}, {'first..."


In [44]:
#initialise pyterrier
if not pt.started():
    pt.init()

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2708/2275133231.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.10 (build: craigm 2024-08-22 17:33), helper_version=0.0.8]
/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2708/2275133231.py:3: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [50]:
#pyterrier exspects a docno field
df_filtered['docno'] = df_filtered['id']
df_filtered['text'] = df_filtered['label']

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2708/3663703777.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['docno'] = df_filtered['id']
/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2708/3663703777.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['text'] = df_filtered['label']
